# Módulos estándar · Sincronización de hilos

Continuación de *Concurrencia* con las primitivas de sincronización de `threading`: `RLock`, `Semaphore`, `Event`, `Condition`, `Barrier`, `Timer` y `threading.local`, más dos problemas clásicos (interbloqueo y filósofos comensales) y una cola de prioridad con apagado ordenado. Como el orden de los hilos varía entre ejecuciones, los ejercicios comprueban propiedades (máximos, totales, orden de fases) en lugar de imprimir trazas que cambian.

## 1. `RLock`: cerrojo reentrante

Una función recursiva que toma un cerrojo y se llama a sí misma se bloquea con un `Lock` normal, porque el mismo hilo intenta adquirirlo dos veces. Comprobarlo con `acquire(timeout=...)` y resolverlo con un `RLock`, que el hilo propietario puede adquirir varias veces.

In [1]:
import random
import threading
import time
from collections import Counter


def suma_recursiva(n, cerrojo):
    if not cerrojo.acquire(timeout=0.2):
        raise TimeoutError(f"no se pudo adquirir el cerrojo en n={n}")
    try:
        return 0 if n == 0 else n + suma_recursiva(n - 1, cerrojo)
    finally:
        cerrojo.release()


try:
    suma_recursiva(5, threading.Lock())
except TimeoutError as e:
    print("Con Lock:", e)
print("Con RLock:", suma_recursiva(5, threading.RLock()))

Con Lock: no se pudo adquirir el cerrojo en n=4
Con RLock: 15


## 2. `Semaphore`: limitar la concurrencia

Simular 12 descargas en hilos, pero permitir como mucho 3 a la vez con un `Semaphore(3)`. Llevar la cuenta de cuántas descargas hay activas en cada momento y comprobar que el máximo nunca pasa de 3.

In [2]:
limite = threading.Semaphore(3)
cuenta = threading.Lock()
activas = maximo = 0


def descargar(i):
    global activas, maximo
    with limite:
        with cuenta:
            activas += 1
            maximo = max(maximo, activas)
        time.sleep(0.05)
        with cuenta:
            activas -= 1


hilos = [threading.Thread(target=descargar, args=(i,)) for i in range(12)]
inicio = time.perf_counter()
for h in hilos:
    h.start()
for h in hilos:
    h.join()
print(f"Máximo de descargas simultáneas: {maximo}; tiempo: {time.perf_counter() - inicio:.2f} s (12 × 0,05 s en tandas de 3)")
assert maximo == 3

Máximo de descargas simultáneas: 3; tiempo: 0.20 s (12 × 0,05 s en tandas de 3)


## 3. `Event`: dar la salida

Preparar 5 corredores en hilos que esperan con `Event.wait()` a que se dé la salida. Comprobar que ninguno arranca antes de `set()` y que todos lo hacen después, guardando el instante en que empieza cada uno.

In [3]:
salida = threading.Event()
arranques = {}


def corredor(nombre):
    salida.wait()
    arranques[nombre] = time.perf_counter()


corredores = [threading.Thread(target=corredor, args=(f"dorsal {i}",)) for i in range(1, 6)]
for c in corredores:
    c.start()
time.sleep(0.1)
print("Antes de la salida han arrancado:", len(arranques))
instante_salida = time.perf_counter()
salida.set()
for c in corredores:
    c.join()
print("Después:", len(arranques), "corredores")
assert len(arranques) == 5 and min(arranques.values()) >= instante_salida

Antes de la salida han arrancado: 0
Después: 5 corredores


## 4. `Condition`: búfer acotado

Implementar una clase `Bufer` de capacidad fija con un `Condition`: `poner` espera mientras está lleno y `sacar` espera mientras está vacío (usando `wait_for`), y cada operación avisa a los demás con `notify_all`. Probarlo con dos productores y dos consumidores y comprobar que se consumen todos los elementos una sola vez y que el búfer nunca supera su capacidad.

In [4]:
from collections import deque


class Bufer:
    def __init__(self, capacidad):
        self.capacidad = capacidad
        self.elementos = deque()
        self.condicion = threading.Condition()
        self.maximo = 0

    def poner(self, x):
        with self.condicion:
            self.condicion.wait_for(lambda: len(self.elementos) < self.capacidad)
            self.elementos.append(x)
            self.maximo = max(self.maximo, len(self.elementos))
            self.condicion.notify_all()

    def sacar(self):
        with self.condicion:
            self.condicion.wait_for(lambda: self.elementos)
            x = self.elementos.popleft()
            self.condicion.notify_all()
            return x


bufer = Bufer(capacidad=4)
consumidos = []
consumidos_cerrojo = threading.Lock()


def productor(inicio):
    for x in range(inicio, inicio + 50):
        bufer.poner(x)


def consumidor():
    while (x := bufer.sacar()) is not None:
        with consumidos_cerrojo:
            consumidos.append(x)


productores = [threading.Thread(target=productor, args=(i,)) for i in (0, 100)]
consumidores = [threading.Thread(target=consumidor) for _ in range(2)]
for h in productores + consumidores:
    h.start()
for h in productores:
    h.join()
for _ in consumidores:
    bufer.poner(None)
for h in consumidores:
    h.join()
print(len(consumidos), "consumidos; ocupación máxima del búfer:", bufer.maximo)
assert sorted(consumidos) == list(range(50)) + list(range(100, 150)) and bufer.maximo <= 4

100 consumidos; ocupación máxima del búfer: 4


## 5. `Barrier`: trabajo por fases

Cuatro hilos hacen un trabajo en tres fases y ninguno puede empezar una fase hasta que todos hayan terminado la anterior. Sincronizarlos con un `Barrier(4)`, registrar en una lista cada `(fase, hilo)` al terminar y comprobar que todas las entradas de una fase aparecen antes que las de la siguiente.

In [5]:
barrera = threading.Barrier(4)
registro = []
registro_cerrojo = threading.Lock()
random.seed(3)
pausas = {(h, f): random.uniform(0, 0.03) for h in range(4) for f in range(3)}


def trabajador(h):
    for fase in range(3):
        time.sleep(pausas[h, fase])
        with registro_cerrojo:
            registro.append((fase, h))
        barrera.wait()


hilos = [threading.Thread(target=trabajador, args=(h,)) for h in range(4)]
for h in hilos:
    h.start()
for h in hilos:
    h.join()
fases = [f for f, _ in registro]
print("Fases en orden de llegada:", fases)
assert fases == sorted(fases) and Counter(fases) == {0: 4, 1: 4, 2: 4}

Fases en orden de llegada: [0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2]


## 6. `Timer`: ejecutar más tarde

Programar con `threading.Timer` dos recordatorios, uno a los 0,1 s y otro a los 0,3 s, y cancelar el segundo antes de que salte. Comprobar que solo se ha ejecutado el primero y que lo ha hecho después de su retraso.

In [6]:
avisos = []
inicio = time.perf_counter()
primero = threading.Timer(0.1, lambda: avisos.append(("tomar agua", time.perf_counter() - inicio)))
segundo = threading.Timer(0.3, lambda: avisos.append(("estirar", time.perf_counter() - inicio)))
primero.start()
segundo.start()
time.sleep(0.15)
segundo.cancel()
primero.join()
segundo.join()
print([(texto, round(t, 1)) for texto, t in avisos])
assert [texto for texto, _ in avisos] == ["tomar agua"] and avisos[0][1] >= 0.1

[('tomar agua', 0.1)]


## 7. Datos propios de cada hilo con `threading.local`

Simular un servidor que atiende peticiones en varios hilos y guarda el usuario de la petición en curso en un objeto `threading.local`, de modo que una función auxiliar pueda leerlo sin recibirlo como argumento. Comprobar que cada hilo ve solo su propio usuario aunque se ejecuten a la vez.

In [7]:
contexto = threading.local()
vistos = {}


def registrar(mensaje):
    return f"[{contexto.usuario}] {mensaje}"


def atender(usuario):
    contexto.usuario = usuario
    time.sleep(random.uniform(0, 0.02))
    vistos[usuario] = registrar("petición atendida")


usuarios = ["ana", "luis", "carmen", "pedro", "yaiza"]
hilos = [threading.Thread(target=atender, args=(u,)) for u in usuarios]
for h in hilos:
    h.start()
for h in hilos:
    h.join()
for u in usuarios:
    print(vistos[u])
assert all(vistos[u].startswith(f"[{u}]") for u in usuarios)
print("¿El hilo principal tiene usuario?", hasattr(contexto, "usuario"))
assert not hasattr(contexto, "usuario")

[ana] petición atendida
[luis] petición atendida
[carmen] petición atendida
[pedro] petición atendida
[yaiza] petición atendida
¿El hilo principal tiene usuario? False


## 8. Interbloqueo

Dos hilos transfieren dinero entre las mismas dos cuentas en sentidos opuestos, y cada uno bloquea primero la cuenta de origen y luego la de destino: si cada uno consigue el primer cerrojo, se esperan para siempre (*deadlock*). Detectarlo con `acquire(timeout=...)` y evitarlo bloqueando siempre las cuentas en el mismo orden (por identificador). Comprobar que el dinero total se conserva.

In [8]:
class Cuenta:
    def __init__(self, ident, saldo):
        self.ident, self.saldo, self.cerrojo = ident, saldo, threading.Lock()


def transferir(origen, destino, cantidad, ordenado, bloqueos):
    primero, segundo = sorted((origen, destino), key=lambda c: c.ident) if ordenado else (origen, destino)
    with primero.cerrojo:
        time.sleep(0.05)
        if not segundo.cerrojo.acquire(timeout=0.3):
            bloqueos.append(f"{origen.ident}->{destino.ident}")
            return
        try:
            origen.saldo -= cantidad
            destino.saldo += cantidad
        finally:
            segundo.cerrojo.release()


for ordenado in (False, True):
    a, b, bloqueos = Cuenta("A", 100), Cuenta("B", 100), []
    hilos = [threading.Thread(target=transferir, args=(a, b, 30, ordenado, bloqueos)),
             threading.Thread(target=transferir, args=(b, a, 10, ordenado, bloqueos))]
    for h in hilos:
        h.start()
    for h in hilos:
        h.join()
    print(f"{'Orden fijo' if ordenado else 'Sin orden'}: transferencias bloqueadas {sorted(bloqueos)}, saldos A={a.saldo} B={b.saldo}")
    assert a.saldo + b.saldo == 200
assert not bloqueos and (a.saldo, b.saldo) == (80, 120)

Sin orden: transferencias bloqueadas ['B->A'], saldos A=70 B=130
Orden fijo: transferencias bloqueadas [], saldos A=80 B=120


## 9. Los filósofos comensales

Cinco filósofos se sientan alrededor de una mesa con un tenedor entre cada dos; para comer necesitan los dos tenedores de sus lados. Resolver el problema sin interbloqueo haciendo que cada filósofo coja primero el tenedor de número menor. Cada uno come 20 veces; comprobar que todos han comido sus 20 veces y que dos vecinos nunca comen a la vez.

In [9]:
N = 5
tenedores = [threading.Lock() for _ in range(N)]
comiendo = [False] * N
comidas = [0] * N
estado = threading.Lock()
conflictos = []


def filosofo(i):
    izquierdo, derecho = i, (i + 1) % N
    for _ in range(20):
        primero, segundo = sorted((izquierdo, derecho))
        with tenedores[primero], tenedores[segundo]:
            with estado:
                if comiendo[(i - 1) % N] or comiendo[(i + 1) % N]:
                    conflictos.append(i)
                comiendo[i] = True
            time.sleep(0.001)
            with estado:
                comiendo[i] = False
                comidas[i] += 1


filosofos = [threading.Thread(target=filosofo, args=(i,)) for i in range(N)]
for f in filosofos:
    f.start()
for f in filosofos:
    f.join()
print("Comidas por filósofo:", comidas, "| conflictos:", len(conflictos))
assert comidas == [20] * N and not conflictos

Comidas por filósofo: [20, 20, 20, 20, 20] | conflictos: 0


## 10. Cola de prioridad y apagado ordenado

Repartir tareas con prioridad entre tres trabajadores usando `queue.PriorityQueue`: cada tarea es `(prioridad, número, descripción)`. Esperar a que se procesen todas con `task_done` y `join`, y después apagar los trabajadores metiendo un centinela por cada uno. Comprobar que con un solo trabajador las tareas salen en orden de prioridad.

In [10]:
import queue


def repartir(tareas, n_trabajadores):
    cola = queue.PriorityQueue()
    hechas = []
    cerrojo = threading.Lock()

    def trabajador():
        while True:
            prioridad, _, descripcion = cola.get()
            if descripcion is None:
                cola.task_done()
                return
            with cerrojo:
                hechas.append((prioridad, descripcion))
            cola.task_done()

    for t in tareas:
        cola.put(t)
    trabajadores = [threading.Thread(target=trabajador) for _ in range(n_trabajadores)]
    for t in trabajadores:
        t.start()
    cola.join()
    for i in range(n_trabajadores):
        cola.put((float("inf"), i, None))
    for t in trabajadores:
        t.join()
    return hechas


random.seed(7)
tareas = [(random.randint(1, 5), i, f"tarea {i}") for i in range(15)]
uno = repartir(tareas, 1)
print("Un trabajador:", [p for p, _ in uno])
assert [p for p, _ in uno] == sorted(p for p, _, _ in tareas)
tres = repartir(tareas, 3)
print("Tres trabajadores procesan", len(tres), "tareas")
assert sorted(tres) == sorted((p, d) for p, _, d in tareas)

Un trabajador: [1, 1, 1, 1, 1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 5]
Tres trabajadores procesan 15 tareas
